# 严格过去验证的Ridge正则化

稳定超额目标继续。上一信息实验训练内改善却在9/11个后续年份恶化；本轮只检验嵌套时间验证是否能拒绝过拟合的固定Ridge10。全部参数先登记于[plan.json](plan.json)，保留H5/25价格特征/费用/持有期及风险门。

In [1]:
from pathlib import Path
import sys,os,json
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/"AGENTS.md").exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
CARD=ROOT/"research"/"purged_ridge_regularization_20261009"
PLAN=json.loads((CARD/"plan.json").read_text())
import qlib
from qlib.data import D
from qlib.workflow import R
qlib.init(provider_uri=str(ROOT/PLAN["source"]["provider_uri"]),region="us",kernels=1)
calendar=D.calendar(freq="day",future=False)
print("Native calendar",calendar[0],calendar[-1],len(calendar))
print("Inner candidates",PLAN["alphas"],"plus explicit historical mean")

Native calendar 2010-03-11 00:00:00 2026-09-25 00:00:00 4162
Inner candidates [10.0, 100.0, 1000.0] plus explicit historical mean


## 训练、验证与外层分工

外层Y年：内层模型只训练至Y−1年开始前，验证Y−1年中在Y年开始前已成熟的H5标签；两资产误差除以内层训练方差再等权。选好后才用Y年前可见数据重拟合。历史均值预测不是全仓SOXX组合；即使选择prior，仍独立计算配置、费用和收益。

In [2]:
from src.workflow.purged_ridge_regularization import run_purged_ridge_regularization
summary,tables,conclusion=run_purged_ridge_regularization(source_root=ROOT,evidence_root=ROOT,card_dir=CARD,recorder_uri=CARD/"qlib_recorder")
print({k:summary[k] for k in ["recorder_id","selected_by_year","historical_gate_pass","objective_achieved"]})

backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native nested Ridge path passed: fixed_ridge10, 10bp


backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native nested Ridge path passed: validation_selected, 10bp


...

backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native nested Ridge path passed: fixed_ridge10, 25bp


backtest loop:   0%|          | 0/2696 [00:00<?, ?it/s]

Native nested Ridge path passed: validation_selected, 25bp


{'recorder_id': 'd46772566a2947859faee38a5d4e08b8', 'selected_by_year': {'2016': 'prior', '2017': 'ridge_1000', '2018': 'prior', '2019': 'ridge_1000', '2020': 'prior', '2021': 'ridge_1000', '2022': 'ridge_10', '2023': 'prior', '2024': 'prior', '2025': 'prior', '2026': 'ridge_1000'}, 'historical_gate_pass': False, 'objective_achieved': False}


In [3]:
from IPython.display import display,Markdown
display(tables["selection_scores.csv"])
display(tables["forecast_increments.csv"])
display(tables["backtest_benchmark_comparison.csv"][["strategy","cost_bps","window","relative_wealth","max_drawdown_pct","cvar5_pct","turnover"]])
display(tables["gates.csv"])
display(tables["parent_control_reproduction.csv"])

,candidate_name,model_kind,ridge_alpha,normalized_mse,mse_soxl,mse_soxx,normalized_mse_soxl,normalized_mse_soxx,inner_training_frame_sha256,validation_rows,exact_tie_priority,selected
0,prior,historical_mean,NaN,0.912130,0.008152,0.000903,0.914036,0.910223,60f0a4059273b3682aa0ddd3388cc763e8bf84a53a15d7...,246,0,True
1,ridge_10,ridge,10.0,1.000874,0.008956,0.000990,1.004159,0.997589,60f0a4059273b3682aa0ddd3388cc763e8bf84a53a15d7...,246,3,False
2,ridge_100,ridge,100.0,0.989821,0.008851,0.000980,0.992340,0.987301,60f0a4059273b3682aa0ddd3388cc763e8bf84a53a15d7...,246,2,False
3,ridge_1000,ridge,1000.0,0.936864,0.008373,0.000928,0.938761,0.934967,60f0a4059273b3682aa0ddd3388cc763e8bf84a53a15d7...,246,1,False
4,prior,historical_mean,NaN,1.016435,0.009059,0.000972,1.034374,0.998497,3fbd5db9caeb09b4cfcc0016048a11045075fe8b858e22...,246,0,False
5,ridge_10,ridge,10.0,1.025149,0.009120,0.000982,1.041422,1.008876,3fbd5db9caeb09b4cfcc0016048a11045075fe8b858e22...,246,3,False
6,ridge_100,ridge,100.0,1.009437,0.008976,0.000968,1.024978,0.993895,3fbd5db9caeb09b4cfcc0016048a11045075fe8b858e22...,246,2,False
7,ridge_1000,ridge,1000.0,0.998794,0.008888,0.000957,1.014885,0.982702,3fbd5db9caeb09b4cfcc0016048a11045075fe8b858e22...,246,1,True
8,prior,historical_mean,NaN,0.573422,0.005152,0.000561,0.578264,0.568580,4595e262b21cce40d02d6eae393d67fe191fbf4c604a16...,245,0,True
9,ridge_10,ridge,10.0,0.713673,0.006430,0.000697,0.721627,0.705720,4595e262b21cce40d02d6eae393d67fe191fbf4c604a16...,245,3,False


,window,rows,start_signal,end_signal,evidence_scope,asset,reference,mse_improvement_reference_minus_full,relative_mse_improvement_pct,squared_bias_improvement,error_variance_improvement,pearson_ic_change,spearman_ic_change
0,all,2692,2016-01-04,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxl,base,0.000419,2.344016,-1.633060e-05,4.357243e-04,-0.027148,-0.008826
1,all,2692,2016-01-04,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxl,prior,-0.000196,-1.134079,1.300625e-05,-2.089385e-04,0.038327,0.040533
2,all,2692,2016-01-04,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxx,base,0.000031,1.597578,-2.834496e-06,3.379401e-05,-0.044008,-0.006921
3,all,2692,2016-01-04,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxx,prior,-0.000021,-1.105997,2.836368e-06,-2.369639e-05,0.024521,0.027014
4,development,1762,2016-01-04,2022-12-30,reused_history_diagnostic_not_forward_or_promo...,soxl,base,0.000431,2.818151,1.875119e-06,4.288152e-04,-0.014802,-0.007260
5,development,1762,2016-01-04,2022-12-30,reused_history_diagnostic_not_forward_or_promo...,soxl,prior,-0.000207,-1.410572,6.284796e-06,-2.128694e-04,0.085185,0.091290
6,development,1762,2016-01-04,2022-12-30,reused_history_diagnostic_not_forward_or_promo...,soxx,base,0.000033,1.959566,-8.570641e-08,3.295704e-05,-0.037911,-0.010711
7,development,1762,2016-01-04,2022-12-30,reused_history_diagnostic_not_forward_or_promo...,soxx,prior,-0.000026,-1.613867,1.561926e-06,-2.768218e-05,0.074400,0.079205
8,reused_diagnostic,930,2023-01-03,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxl,base,0.000398,1.742831,-4.801452e-05,4.460055e-04,-0.051094,-0.020205
9,reused_diagnostic,930,2023-01-03,2026-09-17,reused_history_diagnostic_not_forward_or_promo...,soxl,prior,-0.000176,-0.789456,-1.851926e-05,-1.572306e-04,-0.043131,-0.071647


,strategy,cost_bps,window,relative_wealth,max_drawdown_pct,cvar5_pct,turnover
0,soxx,10.0,development,1.000000,-47.373290,-4.416437,9.990010e-01
1,soxx,10.0,reused_diagnostic,1.000000,-42.789189,-5.295891,0.000000e+00
2,fixed_ridge10,10.0,development,0.871152,-41.213549,-4.385856,2.911196e+01
3,fixed_ridge10,10.0,reused_diagnostic,0.797533,-42.789189,-5.295891,1.463242e+01
4,validation_selected,10.0,development,0.823950,-41.213549,-4.373946,1.539980e+01
5,validation_selected,10.0,reused_diagnostic,1.000000,-42.789189,-5.295891,0.000000e+00
6,soxx,25.0,development,1.000000,-47.373290,-4.418079,9.975062e-01
7,soxx,25.0,reused_diagnostic,1.000000,-42.789189,-5.295891,0.000000e+00
8,fixed_ridge10,25.0,development,0.838612,-41.389645,-4.343586,2.049526e+01
9,fixed_ridge10,25.0,reused_diagnostic,0.978386,-42.789189,-5.295891,1.997506e+00


,cost_bps,aggregate_positive_net_log_excess,minimum_positive_signal_years,median_annual_log_excess_positive,leave_any_one_year_out_log_excess_positive,anchored_max_drawdown_not_worse_than_soxx,cvar5_not_worse_than_soxx,minimum_rolling_positive_excess_rate,stationary_block_20_lower_bound_positive,stationary_block_60_lower_bound_positive,all_pass,positive_years,years,minimum_positive_years,strategy
0,10.0,False,False,False,False,True,True,False,False,False,False,2,7,6,validation_selected
1,25.0,False,False,False,False,True,True,False,False,False,False,2,7,6,validation_selected


,cost_bps,parent_recorder_id,path_sha256,maximum_difference,tolerance,passed
0,10.0,64610b843175452b80e966c5dd99adef,143c60bec31992711d8e19cd11116d09481647cb299580...,0.0,1.000000e-10,True
1,25.0,64610b843175452b80e966c5dd99adef,143c60bec31992711d8e19cd11116d09481647cb299580...,0.0,1.000000e-10,True


## 中文结论

输出与保存的Qlib证据一致；不把本轮失败归档当作完成用户目标。

In [4]:
display(Markdown(conclusion))

# 严格过去验证选择Ridge正则化

本轮保留25价格特征、H5标签与5日执行，只改变收益头正则选择。每年用更早训练期拟合10/100/1000及历史均值预测，以前一年严格成熟验证误差选择，再用当前年开始前成熟数据重拟合。

25bp比较如下；完整开发年2016–2022，其后仍为复用历史：

- development/soxx：收益325.11%，相对SOXX终值1.0000，MaxDD -47.37%，CVaR5 -4.42%，换手1.00。
- reused_diagnostic/soxx：收益398.62%，相对SOXX终值1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。
- development/fixed_ridge10：收益256.50%，相对SOXX终值0.8386，MaxDD -41.39%，CVaR5 -4.34%，换手20.50。
- reused_diagnostic/fixed_ridge10：收益387.84%，相对SOXX终值0.9784，MaxDD -42.79%，CVaR5 -5.30%，换手2.00。
- development/validation_selected：收益250.56%，相对SOXX终值0.8246，MaxDD -41.39%，CVaR5 -4.36%，换手11.62。
- reused_diagnostic/validation_selected：收益398.62%，相对SOXX终值1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。

历史门通过：False；年度选择：{'2016': 'prior', '2017': 'ridge_1000', '2018': 'prior', '2019': 'ridge_1000', '2020': 'prior', '2021': 'ridge_1000', '2022': 'ridge_10', '2023': 'prior', '2024': 'prior', '2025': 'prior', '2026': 'ridge_1000'}。整体目标仍在进行，不晋升。

历史均值预测与全仓SOXX组合是两个不同对照；若选择prior，仍须看真实优化权重、费用与收益，不能称其学到了alpha。内层两资产误差按各自内层训练方差归一，避免SOXL主导，绝不按外层收益选择。

年度/窗口差异与所有44个内层候选分数保留于selection_scores、selection_audit、annual_metrics。预测MSE/IC及组合费用后稳定性分开评价，没有根据后验结果翻号、改alpha或改期限。

全部历史已复用；先前年度验证仍可能不适合下一年。H5重叠、价格口径/次来源观察、历史开盘、连续份额与现金零收益限制保留。

若选择修正仍未通过，下一步依据选择结果、预测增量与真实持仓差异继续定位信息或配置目标的问题；本批归档不关闭稳定超额目标。

Qlib Recorder：d46772566a2947859faee38a5d4e08b8；as_of=2026-09-25。


# 严格过去验证选择Ridge正则化

本轮保留25价格特征、H5标签与5日执行，只改变收益头正则选择。每年用更早训练期拟合10/100/1000及历史均值预测，以前一年严格成熟验证误差选择，再用当前年开始前成熟数据重拟合。

25bp比较如下；完整开发年2016–2022，其后仍为复用历史：

- development/soxx：收益325.11%，相对SOXX终值1.0000，MaxDD -47.37%，CVaR5 -4.42%，换手1.00。
- reused_diagnostic/soxx：收益398.62%，相对SOXX终值1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。
- development/fixed_ridge10：收益256.50%，相对SOXX终值0.8386，MaxDD -41.39%，CVaR5 -4.34%，换手20.50。
- reused_diagnostic/fixed_ridge10：收益387.84%，相对SOXX终值0.9784，MaxDD -42.79%，CVaR5 -5.30%，换手2.00。
- development/validation_selected：收益250.56%，相对SOXX终值0.8246，MaxDD -41.39%，CVaR5 -4.36%，换手11.62。
- reused_diagnostic/validation_selected：收益398.62%，相对SOXX终值1.0000，MaxDD -42.79%，CVaR5 -5.30%，换手0.00。

历史门通过：False；年度选择：{'2016': 'prior', '2017': 'ridge_1000', '2018': 'prior', '2019': 'ridge_1000', '2020': 'prior', '2021': 'ridge_1000', '2022': 'ridge_10', '2023': 'prior', '2024': 'prior', '2025': 'prior', '2026': 'ridge_1000'}。整体目标仍在进行，不晋升。

历史均值预测与全仓SOXX组合是两个不同对照；若选择prior，仍须看真实优化权重、费用与收益，不能称其学到了alpha。内层两资产误差按各自内层训练方差归一，避免SOXL主导，绝不按外层收益选择。

年度/窗口差异与所有44个内层候选分数保留于selection_scores、selection_audit、annual_metrics。预测MSE/IC及组合费用后稳定性分开评价，没有根据后验结果翻号、改alpha或改期限。

全部历史已复用；先前年度验证仍可能不适合下一年。H5重叠、价格口径/次来源观察、历史开盘、连续份额与现金零收益限制保留。

若选择修正仍未通过，下一步依据选择结果、预测增量与真实持仓差异继续定位信息或配置目标的问题；本批归档不关闭稳定超额目标。

Qlib Recorder：d46772566a2947859faee38a5d4e08b8；as_of=2026-09-25。

## 失败归因与下一机制

选择修正将开发预测MSE相对固定10改善SOXL2.82%、SOXX1.96%，仍分别差于过去均值1.41%、1.61%。25bp开发组合终值比0.8246低于固定10的0.8386，2023以后完全等同SOXX；预测误差改善没有自动转成稳定净超额。

所选方案2016初始买入约33.05%SOXL、66.95%cash，并到2021-02-25才首次精确SOXX；前1295个区间相对财富约0.7415。其现金多却仍复制接近1x市场风险，不能泛称空仓择时。开发净log超额−0.19282中毛贡献−0.16625、增量费用−0.02657；杠杆产品残差和长期资产形状是关键。

固定首个决策的预期毛H5收益以SOXX更高，但SOXL/cash入场省约16.68bp；若仅按同一预测估计未来转回SOXX的费用，约需33.28bp，可抵消这项短规划优势。下一实验将冻结全部预测，仅检验benchmark-terminal转换估值；这不是已证明有效，也不会虚扣未来未执行费用。

## 数值与验证边界

修正父引用manifest/hash和nonfinite检查后，完整原配方重验；两次运行的选择、模型、预测与路径逐对象一致，没有改参数或获得新的独立样本。原运行448efe19737f433bb4fae407ed860f36和当前d46772566a2947859faee38a5d4e08b8均保留，当前为引用检查加固版本。158项相关测试、153项独立读回检查通过；6条2696期原生路径重新估值误差为0，账本最大差约5.57e-12，2160求解证书通过。

开发期2/7正年确为2017、2022的经济增益；后段2023/2024/2026的正log差仅约1e-18～1e-16，为等同SOXX时的浮点误差。旧原始统计和失败门不改；下一登记方案用累计1e-8数值分辨下限，不能让这些正零冒充alpha。详情见diagnostic_numerical_floor.md。

全仓仍有15个既有归档重名收集冲突；规定旧Qlib套件8通过、61个既有缺失卡片失败。总体目标保持active。
